# K11 · SinDDM baseline: training + sampling on 2 GPUs (step 4)

Trains SinDDM on the 3 OOD images (128 px), **all three at the same time** (two on GPU 0, one on GPU 1),
for `STEPS = 20000` each (the full 120k-step schedule would take ~24 T4-h per image), with the LR milestones
compressed in proportion. Then samples 30 images from the 5k / 10k / 20k-step checkpoints (a quality-vs-training-time
curve) and from the 3 **official pretrained** models (converged reference).

Progress: one bar per image (training steps, with loss and speed), then one bar per sampling job.
Checkpoints are saved every 10 min, so an interrupted run resumes (attach the output, run again).

**How to run (Kaggle):**
1. Right panel → *Session options*: **Accelerator = GPU T4 x2**, **Internet = On**.
2. Top right **Save Version** → *Save & Run All (Commit)* → Save. It runs in the background (you can close the tab);
   watch it under *View Active Events* / the version's **Logs** (progress lines every 2 min).
3. When it finishes, the result is the version's **Output**: `cs787_k11.zip`.
4. **Not finished?** (it stops itself before Kaggle's 12-h limit) → *Add Input* → *Your Work* → this notebook's
   latest version → then *Save Version* again. It continues where it stopped.

In [ ]:
import time
T_START = time.time()                      # session clock: work stops before SESSION_HOURS
NOTEBOOK = "K11_sinddm"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
SESSION_HOURS = 11.0                       # Kaggle stops sessions at 12 h; we stop starting new work at this point

IMAGES       = ["sd_b0", "sd_fp_house2", "sd_pat_p4"]   # data/sinddm/*.png, 128 px
STEPS        = 20000
SAVE_EVERY   = 5000
SAMPLE_AFTER = [1, 2, 4]      # checkpoints to sample: 1..4 = 5k..20k steps
N_SAMPLES    = 30
SLOTS        = 3              # trainings at the same time (the model is small; 2 share one T4)
SAMPLING_RESERVE_MIN = 75     # session minutes kept for sampling + zipping

In [ ]:
import os, sys, subprocess, shutil, glob, json
if not os.path.isdir("/kaggle/working"):
    raise SystemExit("This notebook is written for Kaggle (kaggle.com/code).")
WORK, TMP = "/kaggle/working", "/tmp/cs787"
PROJECT = f"{TMP}/project"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(TMP, exist_ok=True)
r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, GITHUB_REPO, PROJECT],
                   capture_output=True, text=True)
if r.returncode:
    raise SystemExit("Could not download the code from GitHub. Turn Internet ON: right panel -> Session options -> "
                     "Internet (needs a phone-verified Kaggle account), then run again.\n" + r.stderr)
sys.path.insert(0, f"{PROJECT}/src")
os.environ["PYTHONPATH"] = f"{PROJECT}/src" + os.pathsep + os.environ.get("PYTHONPATH", "")
from sidx.kaggle import gather_inputs, unpack_own, pack
from sidx.progress import Bars
print("code:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
gather_inputs()          # results of earlier versions attached under /kaggle/input (resume)

In [ ]:
import torch
N_GPUS = torch.cuda.device_count()
if N_GPUS == 0:
    raise SystemExit("No GPU. Right panel -> Session options -> Accelerator -> 'GPU T4 x2', then run again.")
cap = torch.cuda.get_device_capability(0)
if f"sm_{cap[0]}{cap[1]}" not in torch.cuda.get_arch_list():
    raise SystemExit(f"{torch.cuda.get_device_name(0)} is not supported by this PyTorch build: choose 'GPU T4 x2' (not P100).")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", N_GPUS, "GPU(s) -> all of them are used")

In [ ]:
SINDDM = f"{TMP}/SinDDM"
if not os.path.exists(SINDDM):
    subprocess.run(["git", "clone", "-q", "https://github.com/fallenshock/SinDDM", SINDDM], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", SINDDM_COMMIT], check=True)
shutil.copytree(f"{PROJECT}/patches/sinddm_port", SINDDM, dirs_exist_ok=True, ignore=shutil.ignore_patterns("CHANGES.diff"))
!pip install -q einops ftfy scikit-image
print(subprocess.run(["git", "-C", SINDDM, "status", "--short"], capture_output=True, text=True).stdout)

## Inputs and resume state

In [ ]:
import collections
from sidx.gpupool import Task, run_pool
unpack_own()
DATA = f"{TMP}/sinddm_data"
for i in IMAGES:
    os.makedirs(f"{DATA}/{i}", exist_ok=True)
    shutil.copy(f"{PROJECT}/data/sinddm/{i}.png", f"{DATA}/{i}/{i}.png")
CK = f"{WORK}/checkpoints/sinddm"
OUT_ROOT = f"{WORK}/outputs"

def trained_steps(i):
    p = f"{CK}/{i}/{i}/model-last.pt"
    if not os.path.exists(p):
        return 0
    with torch.serialization.safe_globals([collections.Counter]):
        return int(torch.load(p, map_location="cpu", weights_only=True)["step"])

START = {i: trained_steps(i) for i in IMAGES}
print("steps trained so far:", START)

## Train all images in parallel (live bar per image: steps, loss, speed)

In [ ]:
TRAIN_DEADLINE = T_START + SESSION_HOURS * 3600 - SAMPLING_RESERVE_MIN * 60

def train_argv(i):
    def argv():
        budget = max(1.0, (TRAIN_DEADLINE - time.time()) / 60)
        return [sys.executable, "main.py", "--scope", i, "--mode", "train", "--dataset_folder", f"{DATA}/{i}/",
                "--image_name", f"{i}.png", "--results_folder", CK, "--train_num_steps", str(STEPS + 1),
                "--save_and_sample_every", str(SAVE_EVERY), "--sched_k_milestones", "3", "7", "12", "13", "15", "18",
                "--resume", "--time_budget_min", f"{budget:.1f}", "--checkpoint_every_min", "10", "--seed", "0",
                "--skip_final_sample"]
    return argv

def read_progress(i):
    try:
        return json.load(open(f"{CK}/{i}/{i}/progress.json"))
    except Exception:
        return {}

def poll_train():
    return {i: (min(STEPS, max(START[i], read_progress(i).get("step", 0))), STEPS) for i in IMAGES}

def train_postfix():
    out = {}
    for i in IMAGES:
        p = read_progress(i)
        if p.get("loss") is not None:
            out[i] = f"loss {p['loss']:.4f} | {p.get('steps_per_s', 0):.2f} steps/s"
    return out

tasks = [Task(f"train_{i}", train_argv(i), cwd=SINDDM) for i in IMAGES if START[i] < STEPS]
if tasks:
    run_pool(tasks, poll_train, slots=SLOTS, n_gpus=N_GPUS, log_dir=f"{WORK}/logs/{NOTEBOOK}", deadline=TRAIN_DEADLINE,
             unit="step", extra_postfix=train_postfix)
print("steps trained now:", {i: trained_steps(i) for i in IMAGES})

## Sample every finished checkpoint + the 3 pretrained models (30 images each, standard output layout)

In [ ]:
import shlex
def collect_cmd(job, scope, data_dir, image_name, results_folder, milestone, image_id, image_path):
    return [sys.executable, f"{PROJECT}/scripts/sinddm_collect.py", "--sinddm-root", SINDDM, "--scope", scope,
            "--dataset-folder", data_dir, "--image-name", image_name, "--results-folder", results_folder,
            "--milestone", str(milestone), "--out-root", OUT_ROOT, "--job", job, "--image-id", image_id,
            "--image-path", image_path, "--n", str(N_SAMPLES), "--batch", "10"]

expected, tasks = [], []
for i in IMAGES:                     # one task per image (its checkpoints one after another: same scratch folder)
    cmds = []
    for k in SAMPLE_AFTER:
        if os.path.exists(f"{CK}/{i}/{i}/model-{k}.pt"):
            job = f"sinddm_steps{k * SAVE_EVERY}"
            cmds.append(collect_cmd(job, i, f"{DATA}/{i}/", f"{i}.png", CK, k, i, f"{PROJECT}/data/sinddm/{i}.png"))
            expected.append((job, i))
        else:
            print(i, "checkpoint", k * SAVE_EVERY, "not trained yet")
    if cmds:
        tasks.append(Task(f"sample_{i}", ["bash", "-c", " && ".join(shlex.join(c) for c in cmds)], cwd=SINDDM))
for scope in ("forest", "night_sky", "pyramids"):
    tasks.append(Task(f"sample_pretrained_{scope}", collect_cmd("sinddm_pretrained_full", scope, f"{SINDDM}/datasets/{scope}/",
                      f"{scope}.png", f"{SINDDM}/results", "pretrained", scope, f"{SINDDM}/datasets/{scope}/{scope}.png"), cwd=SINDDM))
    expected.append(("sinddm_pretrained_full", scope))

def poll_sample():
    res = {}
    for job, i in expected:
        n = len(glob.glob(f"{OUT_ROOT}/phase8_sinddm/{job}__sinddm/{i}/seed_*.png"))
        d, t = res.get(job, (0, 0))
        res[job] = (d + min(n, N_SAMPLES), t + N_SAMPLES)
    return res

try:
    run_pool(tasks, poll_sample, n_gpus=N_GPUS, log_dir=f"{WORK}/logs/{NOTEBOOK}", unit="image",
             deadline=T_START + SESSION_HOURS * 3600 - 15 * 60)
finally:
    print(poll_sample())
    pack("k11", kinds=("outputs", "checkpoints"))

In [ ]:
os.makedirs(f"{WORK}/logs", exist_ok=True)
!pip freeze > {WORK}/logs/{NOTEBOOK}_pip_freeze.txt
print("finished; output files:", sorted(os.listdir(WORK)))